In [7]:
# --- CONFIGURABLE PARAMETERS (PySpark version) ---
profile = 'healthy'  # Options: 'none', 'healthy', 'high_protein', 'high_fiber'
ingredients = ['banana', 'milk', 'sugar', 'potato', 'cucumber', 'vinegar', 'whole wheat', 'oat', 'spinach', 'broccoli', 'carrot', 'olive oil', 'almond', 'quinoa', 'lentil', 'chicken breast', 'salmon', 'egg',
    'tomato', 'avocado', 'brown rice', 'cauliflower', 'zucchini', 'bell pepper', 'garlic', 'onion', 'sweet potato', 'kale', 'arugula', 'mushroom', 'cod', 'tuna', 'sardine',
    'hazelnut', 'cashew', 'pistachio', 'sunflower seed', 'sesame seed', 'turkey']
kcal_min = None   # e.g. 100
kcal_max = None   # e.g. 300
dislikes = ['palm oil', 'margarine', 'sodium benzoate', 'monosodium glutamate']
top_n = 10        # Number of recipe ideas to return

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set, trim, regexp_replace, udf, size, array_intersect, array, lit
from pyspark.sql import functions as F


In [ ]:
# --- LOAD DATA ---
spark = SparkSession.builder.appName('RecipeIdeas').getOrCreate()
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df_ing = spark.read.schema(ingredients_schema).parquet(
    '../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
)
df_nutri = spark.read.parquet('../output/nutritional_profiles/part-00000-006c6ee8-befb-47d8-8451-e7dae4de0d2f-c000.snappy.parquet')
df = df_ing.join(df_nutri, df_ing.fdc_id == df_nutri.fdc_id, 'inner')
df = df.select(df_ing.fdc_id, df_ing.description, 'all_ingredients', *[c for c in df_nutri.columns if c != 'fdc_id'])

In [9]:
# --- INGREDIENT NORMALIZATION (PySpark) ---
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

def normalize_ingredient_py(ing):
    return re.sub(r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]', '', ing.lower().strip())

def normalize_ingredient_list(lst):
    if lst is None:
        return []
    return [normalize_ingredient_py(i) for i in lst]

normalize_ingredient_list_udf = udf(normalize_ingredient_list, ArrayType(StringType()))
df = df.withColumn('ingredients_norm', normalize_ingredient_list_udf(col('all_ingredients')))


In [ ]:
# --- FILTERING ---
# Prepare dislikes and user ingredients as Spark arrays
user_ingredients_norm = [re.sub(r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]', '', i.lower().strip()) for i in ingredients]
dislikes_norm = [re.sub(r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]', '', d.lower().strip()) for d in dislikes]

# Filter by dislikes
if dislikes_norm:
    df = df.withColumn('has_dislike', F.expr(f"array_intersect(ingredients_norm, array({','.join([f'\'{d}\'' for d in dislikes_norm])}))"))
    df = df.filter(F.size('has_dislike') == 0)

# Filter by kcal
if kcal_min is not None:
    df = df.filter((col('energy').isNotNull()) & (col('energy') >= kcal_min))
if kcal_max is not None:
    df = df.filter((col('energy').isNotNull()) & (col('energy') <= kcal_max))

# Filter by available ingredients: at least min(3, len(ingredients)) matches
if user_ingredients_norm:
    min_match = min(3, len(user_ingredients_norm))
    df = df.withColumn('ingredient_overlap', F.expr(f"size(array_intersect(ingredients_norm, array({','.join([f'\'{i}\'' for i in user_ingredients_norm])})))"))
    df = df.filter(col('ingredient_overlap') >= min_match)


In [ ]:
# --- SCORING FUNCTION ---
from pyspark.sql.types import FloatType
PROFILE_WEIGHTS = {
    'none':     {'fiber': 0,   'protein': 0,   'sugars': 0,   'total_fat': 0,   'energy': 0},
    'healthy':  {'fiber': 1.5, 'protein': 1.2, 'sugars': -1.5, 'total_fat': -0.8, 'energy': -0.01},
    'high_protein': {'fiber': 0, 'protein': 2.0, 'sugars': 0, 'total_fat': 0, 'energy': 0},
    'high_fiber':   {'fiber': 2.0, 'protein': 0, 'sugars': 0, 'total_fat': 0, 'energy': 0}
}
weights = PROFILE_WEIGHTS.get(profile, PROFILE_WEIGHTS['none'])

def score_food_spark(ingredient_overlap, fiber, protein, sugars, total_fat, energy):
    score = 2.0 * ingredient_overlap
    if fiber is not None: score += weights['fiber'] * fiber
    if protein is not None: score += weights['protein'] * protein
    if sugars is not None: score += weights['sugars'] * sugars
    if total_fat is not None: score += weights['total_fat'] * total_fat
    if energy is not None: score += weights['energy'] * energy
    return float(score)

score_food_udf = udf(score_food_spark, FloatType())
df = df.withColumn('recipe_score', score_food_udf('ingredient_overlap', 'fiber', 'protein', 'sugars', 'total_fat', 'energy'))


In [ ]:
# --- MAIN RECIPE SUGGESTION LOGIC ---
from pyspark.sql.window import Window

# Sort and select top N
w = Window.orderBy(col('recipe_score').desc())
df = df.withColumn('row_num', F.row_number().over(w)).filter(col('row_num') <= top_n)

# Show results
result_cols = ['description', 'ingredients_norm', 'energy', 'fiber', 'protein', 'sugars', 'total_fat', 'recipe_score']
df.select(result_cols).show(truncate=False)

26/01/15 18:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/15 18:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/15 18:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/15 18:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/15 18:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
**How it works:**
- Enter your available ingredients, dislikes, kcal range, and nutrition preferences at the top.
- The notebook will suggest foods/recipes that match your configuration, prioritizing ingredient overlap and nutrition profile.
- You can easily adjust the scoring logic for your needs.